In [464]:
import pandas as pd
import re
import plotly.express as px
import numpy as np
from sklearn.preprocessing import MinMaxScaler

### RQ2.1
varying the number of parameters in the given clause

In [465]:
current_result = "RQ1_shared_scenarios_results"

In [466]:
file_path = f'res/outcome/similarities/{current_result}.csv'
raw_df = pd.read_csv(file_path, sep=';')
raw_df

,desired_scenario,cadidate_scenario,Desired Given DNF,Shared Given DNF,Penality Given,Global Similarity Given,Final Similarity Given,Desired When DNF,Shared When DNF,Penality When,Global Similarity When,Final Similarity When,Desired Then DNF,Shared Then DNF,Penality Then,Global Similarity Then,Final Similarity Then,Scenario Similarity,Action
0,Desired_1,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,takeoff == 1,takeoff == 1,0.0,1.0,1.0,height >= 30,height >= 30 AND source_distance == 0,0.090909,1.0,0.909091,0.914141,behavior_3
1,Desired_2,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,takeoff == 1,takeoff == 1,0.0,1.0,1.0,source_distance == 0,height >= 30 AND source_distance == 0,0.090909,1.0,0.909091,0.914141,behavior_3
2,Desired_3,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,takeoff == 1,takeoff == 1,0.0,1.0,1.0,a > 10,height >= 30 AND source_distance == 0,1.000000,0.0,-1.000000,0.277778,behavior_3
3,Desired_4,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,takeoff == 1,takeoff == 1,0.0,1.0,1.0,b > 10,height >= 30 AND source_distance == 0,1.000000,0.0,-1.000000,0.277778,behavior_3
4,Desired_5,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,takeoff == 1,takeoff == 1,0.0,1.0,1.0,height >= 30 AND source_distance == 0,height >= 30 AND source_distance == 0,0.000000,1.0,1.000000,0.944444,behavior_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,Desired_461,candidate,battery >= 10 AND satellite_count >= 7 AND win...,battery >= 10 AND satellite_count >= 7 AND win...,0.375000,1.0,0.625000,takeoff == 1,takeoff == 1,0.0,1.0,1.0,height >= 30 AND source_distance == 0 AND a > 10,height >= 30 AND source_distance == 0,0.310345,1.0,0.689655,0.771552,behavior_3
461,Desired_462,candidate,battery >= 10 AND satellite_count >= 7 AND win...,battery >= 10 AND satellite_count >= 7 AND win...,0.375000,1.0,0.625000,takeoff == 1,takeoff == 1,0.0,1.0,1.0,height >= 30 AND source_distance == 0 AND b > 10,height >= 30 AND source_distance == 0,0.310345,1.0,0.689655,0.771552,behavior_3
462,Desired_463,candidate,battery >= 10 AND satellite_count >= 7 AND win...,battery >= 10 AND satellite_count >= 7 AND win...,0.375000,1.0,0.625000,takeoff == 1,takeoff == 1,0.0,1.0,1.0,height >= 30 AND a > 10 AND b > 10,height >= 30 AND source_distance == 0,0.655172,1.0,0.344828,0.656609,behavior_3
463,Desired_464,candidate,battery >= 10 AND satellite_count >= 7 AND win...,battery >= 10 AND satellite_count >= 7 AND win...,0.375000,1.0,0.625000,takeoff == 1,takeoff == 1,0.0,1.0,1.0,source_distance == 0 AND a > 10 AND b > 10,height >= 30 AND source_distance == 0,0.655172,1.0,0.344828,0.656609,behavior_3


In [467]:
#adaptacao quando a similaridade é negativa, considerar 0
raw_df['Scenario Similarity'] = raw_df['Scenario Similarity'].apply(lambda x: 0 if x < 0 else x)

In [468]:
raw_df.columns

Index(['desired_scenario', 'cadidate_scenario', 'Desired Given DNF',
       'Shared Given DNF', 'Penality Given', 'Global Similarity Given',
       'Final Similarity Given', 'Desired When DNF', 'Shared When DNF',
       'Penality When', 'Global Similarity When', 'Final Similarity When',
       'Desired Then DNF', 'Shared Then DNF', 'Penality Then',
       'Global Similarity Then', 'Final Similarity Then',
       'Scenario Similarity', 'Action'],
      dtype='object')

In [469]:
given_then_df=raw_df[['desired_scenario', 'cadidate_scenario', 'Desired Given DNF',
       'Shared Given DNF', 'Penality Given', 'Global Similarity Given',
       'Final Similarity Given','Desired Then DNF', 'Shared Then DNF', 'Penality Then',
       'Global Similarity Then', 'Final Similarity Then',
       'Scenario Similarity']]

given_then_df = given_then_df.rename(columns={'desired_scenario': 'desired_identifier',
                                               'cadidate_scenario': 'cadidate_identifier',
                                               'Desired Given DNF': 'desired_given_exp',
                                               'Shared Given DNF': 'candidate_given_exp',
                                                'Desired Then DNF': 'desired_then_exp',
                                                'Shared Then DNF': 'candidate_then_exp'})

given_then_df.head()




,desired_identifier,cadidate_identifier,desired_given_exp,candidate_given_exp,Penality Given,Global Similarity Given,Final Similarity Given,desired_then_exp,candidate_then_exp,Penality Then,Global Similarity Then,Final Similarity Then,Scenario Similarity
0,Desired_1,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,height >= 30,height >= 30 AND source_distance == 0,0.090909,1.0,0.909091,0.914141
1,Desired_2,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,source_distance == 0,height >= 30 AND source_distance == 0,0.090909,1.0,0.909091,0.914141
2,Desired_3,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,a > 10,height >= 30 AND source_distance == 0,1.000000,0.0,-1.000000,0.277778
3,Desired_4,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,b > 10,height >= 30 AND source_distance == 0,1.000000,0.0,-1.000000,0.277778
4,Desired_5,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,height >= 30 AND source_distance == 0,height >= 30 AND source_distance == 0,0.000000,1.0,1.000000,0.944444


In [470]:
given_then_df.columns

Index(['desired_identifier', 'cadidate_identifier', 'desired_given_exp',
       'candidate_given_exp', 'Penality Given', 'Global Similarity Given',
       'Final Similarity Given', 'desired_then_exp', 'candidate_then_exp',
       'Penality Then', 'Global Similarity Then', 'Final Similarity Then',
       'Scenario Similarity'],
      dtype='object')

In [471]:
# Extrair variáveis do Given e Then do desejado e do candidato
def extrair_variaveis(expressao):
    variaveis = re.findall(r'(\w+)\s*(?:>=|<=|>|<|==|!=)', expressao)
    return ', '.join(variaveis)  # Opcional: juntar as variáveis em uma string separada por vírgula

# Aplicando a função para cada linha da coluna 'expressao' e armazenando na coluna 'vars'
given_then_df['desired_given_vars'] = given_then_df['desired_given_exp'].apply(extrair_variaveis)
given_then_df['candidate_given_vars'] = given_then_df['candidate_given_exp'].apply(extrair_variaveis)

given_then_df['desired_then_vars'] = given_then_df['desired_then_exp'].apply(extrair_variaveis)
given_then_df['candidate_then_vars'] = given_then_df['candidate_then_exp'].apply(extrair_variaveis)

temp=given_then_df[['desired_given_exp','desired_given_vars','candidate_given_exp','candidate_given_vars','desired_then_exp','desired_then_vars','candidate_then_exp','candidate_then_vars']]
temp

,desired_given_exp,desired_given_vars,candidate_given_exp,candidate_given_vars,desired_then_exp,desired_then_vars,candidate_then_exp,candidate_then_vars
0,battery >= 10,battery,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",height >= 30,height,height >= 30 AND source_distance == 0,"height, source_distance"
1,battery >= 10,battery,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",source_distance == 0,source_distance,height >= 30 AND source_distance == 0,"height, source_distance"
2,battery >= 10,battery,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",a > 10,a,height >= 30 AND source_distance == 0,"height, source_distance"
3,battery >= 10,battery,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",b > 10,b,height >= 30 AND source_distance == 0,"height, source_distance"
4,battery >= 10,battery,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",height >= 30 AND source_distance == 0,"height, source_distance",height >= 30 AND source_distance == 0,"height, source_distance"
...,...,...,...,...,...,...,...,...
460,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed, a, b",battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",height >= 30 AND source_distance == 0 AND a > 10,"height, source_distance, a",height >= 30 AND source_distance == 0,"height, source_distance"
461,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed, a, b",battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",height >= 30 AND source_distance == 0 AND b > 10,"height, source_distance, b",height >= 30 AND source_distance == 0,"height, source_distance"
462,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed, a, b",battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",height >= 30 AND a > 10 AND b > 10,"height, a, b",height >= 30 AND source_distance == 0,"height, source_distance"
463,battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed, a, b",battery >= 10 AND satellite_count >= 7 AND win...,"battery, satellite_count, wind_speed",source_distance == 0 AND a > 10 AND b > 10,"source_distance, a, b",height >= 30 AND source_distance == 0,"height, source_distance"


In [472]:
# Calcular a interseção entre as variáveis do Given e do Then do desejado e do candidato

def calcular_intersecao_given(row):
    # Separar as variáveis em listas
    desired_given = set(row['desired_given_vars'].split(', '))
    candidate_given = set(row['candidate_given_vars'].split(', '))
    
    # Calcular a interseção
    intersecao = desired_given & candidate_given
    return ', '.join(intersecao)  # Juntar as variáveis da interseção em uma string

given_then_df['intersection_given_vars'] = given_then_df.apply(calcular_intersecao_given, axis=1)

def calcular_intersecao_then(row):
    # Separar as variáveis em listas

    desired_then = set(row['desired_then_vars'].split(', '))
    candidate_then = set(row['candidate_then_vars'].split(', '))
    
    # Calcular a interseção
    intersecao = desired_then & candidate_then
    return ', '.join(intersecao)  # Juntar as variáveis da interseção em uma string

# Aplicando a função para cada linha e criando uma nova coluna chamada 'intersection_vars'
given_then_df['intersection_then_vars'] = given_then_df.apply(calcular_intersecao_then, axis=1)


temp=given_then_df[['desired_given_vars','candidate_given_vars','intersection_given_vars','desired_then_vars','candidate_then_vars','intersection_then_vars']]
temp

,desired_given_vars,candidate_given_vars,intersection_given_vars,desired_then_vars,candidate_then_vars,intersection_then_vars
0,battery,"battery, satellite_count, wind_speed",battery,height,"height, source_distance",height
1,battery,"battery, satellite_count, wind_speed",battery,source_distance,"height, source_distance",source_distance
2,battery,"battery, satellite_count, wind_speed",battery,a,"height, source_distance",
3,battery,"battery, satellite_count, wind_speed",battery,b,"height, source_distance",
4,battery,"battery, satellite_count, wind_speed",battery,"height, source_distance","height, source_distance","source_distance, height"
...,...,...,...,...,...,...
460,"battery, satellite_count, wind_speed, a, b","battery, satellite_count, wind_speed","satellite_count, battery, wind_speed","height, source_distance, a","height, source_distance","source_distance, height"
461,"battery, satellite_count, wind_speed, a, b","battery, satellite_count, wind_speed","satellite_count, battery, wind_speed","height, source_distance, b","height, source_distance","source_distance, height"
462,"battery, satellite_count, wind_speed, a, b","battery, satellite_count, wind_speed","satellite_count, battery, wind_speed","height, a, b","height, source_distance",height
463,"battery, satellite_count, wind_speed, a, b","battery, satellite_count, wind_speed","satellite_count, battery, wind_speed","source_distance, a, b","height, source_distance",source_distance


In [473]:
# o que tem de extra no nao referencia (given)
def given_plus(row):
    desired_given = set(row['desired_given_vars'].split(', '))
    candidate_given = set(row['candidate_given_vars'].split(', '))
    
    # Calcular a interseção
    given_plus = desired_given - candidate_given
    return ', '.join(given_plus)

# é o que tem de falta no nao referencia (given)
def given_less(row):
    desired_given = set(row['desired_given_vars'].split(', '))
    candidate_given = set(row['candidate_given_vars'].split(', '))
    
    # Calcular a interseção
    given_less = candidate_given - desired_given
    return ', '.join(given_less)

# o que tem de extra no nao referencia (then)
def then_plus(row):
    desired_then = set(row['desired_then_vars'].split(', '))
    candidate_then = set(row['candidate_then_vars'].split(', '))
    
    # Calcular a interseção
    then_plus = desired_then-candidate_then 
    return ', '.join(then_plus)

# é o que tem de falta no nao referencia (then)
def then_less(row):
    desired_then = set(row['desired_then_vars'].split(', '))
    candidate_then = set(row['candidate_then_vars'].split(', '))
    
    # Calcular a interseção
    then_less = candidate_then-desired_then 
    return ', '.join(then_less)

given_then_df['given_plus'] = given_then_df.apply(given_plus, axis=1)
given_then_df['given_less'] = given_then_df.apply(given_less, axis=1)
given_then_df['then_plus'] = given_then_df.apply(then_plus, axis=1)
given_then_df['then_less'] = given_then_df.apply(then_less, axis=1)

temp=given_then_df[["candidate_given_vars","desired_given_vars","given_plus","given_less","candidate_then_vars","desired_then_vars","then_plus","then_less"]]
temp

,candidate_given_vars,desired_given_vars,given_plus,given_less,candidate_then_vars,desired_then_vars,then_plus,then_less
0,"battery, satellite_count, wind_speed",battery,,"satellite_count, wind_speed","height, source_distance",height,,source_distance
1,"battery, satellite_count, wind_speed",battery,,"satellite_count, wind_speed","height, source_distance",source_distance,,height
2,"battery, satellite_count, wind_speed",battery,,"satellite_count, wind_speed","height, source_distance",a,a,"source_distance, height"
3,"battery, satellite_count, wind_speed",battery,,"satellite_count, wind_speed","height, source_distance",b,b,"source_distance, height"
4,"battery, satellite_count, wind_speed",battery,,"satellite_count, wind_speed","height, source_distance","height, source_distance",,
...,...,...,...,...,...,...,...,...
460,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",,"height, source_distance","height, source_distance, a",a,
461,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",,"height, source_distance","height, source_distance, b",b,
462,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",,"height, source_distance","height, a, b","a, b",source_distance
463,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",,"height, source_distance","source_distance, a, b","a, b",height


In [474]:
#Função para contar quantas variáveis existem em uma string separada por ', '
def contar_variaveis(variaveis_str):
    if variaveis_str.strip() == '':
        return 0
    return len(variaveis_str.split(', '))

# Contando o número de variáveis em 'desired_vars', 'candidate_vars' e 'intersection_vars'
given_then_df['#given_plus'] = given_then_df['given_plus'].apply(contar_variaveis)
given_then_df['#given_less'] = given_then_df['given_less'].apply(contar_variaveis)

given_then_df['#then_plus'] = given_then_df['then_plus'].apply(contar_variaveis)
given_then_df['#then_less'] = given_then_df['then_less'].apply(contar_variaveis)

temp=given_then_df[["candidate_given_vars","desired_given_vars","given_plus","#given_plus","given_less","#given_less","candidate_then_vars","desired_then_vars","then_plus","#then_plus","then_less","#then_less"]]
temp

,candidate_given_vars,desired_given_vars,given_plus,#given_plus,given_less,#given_less,candidate_then_vars,desired_then_vars,then_plus,#then_plus,then_less,#then_less
0,"battery, satellite_count, wind_speed",battery,,0,"satellite_count, wind_speed",2,"height, source_distance",height,,0,source_distance,1
1,"battery, satellite_count, wind_speed",battery,,0,"satellite_count, wind_speed",2,"height, source_distance",source_distance,,0,height,1
2,"battery, satellite_count, wind_speed",battery,,0,"satellite_count, wind_speed",2,"height, source_distance",a,a,1,"source_distance, height",2
3,"battery, satellite_count, wind_speed",battery,,0,"satellite_count, wind_speed",2,"height, source_distance",b,b,1,"source_distance, height",2
4,"battery, satellite_count, wind_speed",battery,,0,"satellite_count, wind_speed",2,"height, source_distance","height, source_distance",,0,,0
...,...,...,...,...,...,...,...,...,...,...,...,...
460,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",2,,0,"height, source_distance","height, source_distance, a",a,1,,0
461,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",2,,0,"height, source_distance","height, source_distance, b",b,1,,0
462,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",2,,0,"height, source_distance","height, a, b","a, b",2,source_distance,1
463,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed, a, b","a, b",2,,0,"height, source_distance","source_distance, a, b","a, b",2,height,1


In [475]:
given_then_df['Scenario Similarity']=given_then_df['Scenario Similarity'].round(3)

In [476]:
temp = given_then_df[['candidate_given_vars','desired_given_vars','#given_plus','#given_less','desired_then_vars','candidate_then_vars','#then_plus','#then_less','Scenario Similarity']].sort_values(by='Scenario Similarity', ascending=False)

temp

,candidate_given_vars,desired_given_vars,#given_plus,#given_less,desired_then_vars,candidate_then_vars,#then_plus,#then_less,Scenario Similarity
229,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed",0,0,"height, source_distance","height, source_distance",0,0,1.000
79,"battery, satellite_count, wind_speed","battery, satellite_count",0,1,"height, source_distance","height, source_distance",0,0,0.984
94,"battery, satellite_count, wind_speed","battery, wind_speed",0,1,"height, source_distance","height, source_distance",0,0,0.984
139,"battery, satellite_count, wind_speed","satellite_count, wind_speed",0,1,"height, source_distance","height, source_distance",0,0,0.984
225,"battery, satellite_count, wind_speed","battery, satellite_count, wind_speed",0,0,height,"height, source_distance",0,1,0.970
...,...,...,...,...,...,...,...,...,...
62,"battery, satellite_count, wind_speed",b,1,3,a,"height, source_distance",1,2,0.000
54,"battery, satellite_count, wind_speed",a,1,3,"a, b","height, source_distance",2,2,0.000
48,"battery, satellite_count, wind_speed",a,1,3,b,"height, source_distance",1,2,0.000
63,"battery, satellite_count, wind_speed",b,1,3,b,"height, source_distance",1,2,0.000


In [477]:
temp = given_then_df[['#given_plus','#given_less','#then_plus','#then_less','Scenario Similarity']].sort_values(by='Scenario Similarity', ascending=False)

temp

,#given_plus,#given_less,#then_plus,#then_less,Scenario Similarity
229,0,0,0,0,1.000
79,0,1,0,0,0.984
94,0,1,0,0,0.984
139,0,1,0,0,0.984
225,0,0,0,1,0.970
...,...,...,...,...,...
62,1,3,1,2,0.000
54,1,3,2,2,0.000
48,1,3,1,2,0.000
63,1,3,1,2,0.000


In [478]:
import plotly.express as px

# Compute the correlation matrix for the specified columns
correlation_matrix = given_then_df[['#given_plus', '#given_less', '#then_plus', '#then_less', 'Scenario Similarity']].corr()

# Generate a heatmap using Plotly
fig = px.imshow(correlation_matrix, 
                text_auto=True, 
                color_continuous_scale='RdBu_r', 
                title="Correlation Matrix")

# Show the plot
fig.show()


Com esse gráfico a gente conclui que é pior erros por falta do que por excesso. Tbm vale a destacar que o then_less é mais impactante pois ele tem menos variaveis do que o given_less, entao cada falta terá um impacto maior.

In [479]:
# given_then_df.drop_duplicates(subset=['#given_plus', '#given_less', '#then_plus', '#then_less'])[['#given_plus','#given_less','#then_plus','#then_less','Scenario Similarity']]

In [480]:
# Função para categorizar o tipo de erro com base nas regras fornecidas
def categorize_error(row):
    # Condição para "no_error"
    if row['#given_plus'] == 0 and row['#then_plus'] == 0 and row['#given_less'] == 0 and row['#then_less'] == 0:
        return 'no_error'
    # Condição para "error_alfa"
    elif row['#given_plus'] == 0 and row['#then_plus'] == 0:
        return 'alfa'
    # Condição para "error_beta"
    elif row['#given_less'] == 0 and row['#then_less'] == 0:
        return 'beta'
    # Caso contrário, é "error_alfa_beta"
    else:
        return 'alfa+beta'

# Aplicar a função para criar a coluna "tipo de erro"
given_then_df['tipo_de_erro'] = given_then_df.apply(categorize_error, axis=1)
temp=given_then_df[['#given_plus','#given_less','#then_plus','#then_less','Scenario Similarity','tipo_de_erro']]
temp


,#given_plus,#given_less,#then_plus,#then_less,Scenario Similarity,tipo_de_erro
0,0,2,0,1,0.914,alfa
1,0,2,0,1,0.914,alfa
2,0,2,1,2,0.278,alfa+beta
3,0,2,1,2,0.278,alfa+beta
4,0,2,0,0,0.944,alfa
...,...,...,...,...,...,...
460,2,0,1,0,0.772,beta
461,2,0,1,0,0.772,beta
462,2,0,2,1,0.657,alfa+beta
463,2,0,2,1,0.657,alfa+beta


In [497]:
%pip install openpyxl

given_then_df.to_excel("given_then_df.xlsx", index=False)

   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---- ----------------------------------- 30.7/250.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------  245.8/250.9 kB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [482]:
given_then_df[["Scenario Similarity","tipo_de_erro"]].sort_values(by='Scenario Similarity', ascending=False).to_csv('test.csv', sep=';', index=False)   

In [483]:
given_then_df = given_then_df[given_then_df['tipo_de_erro'] != 'no_error'].reset_index(drop=True)

# Exibir as primeiras linhas para confirmar a remoção
given_then_df.head()

,desired_identifier,cadidate_identifier,desired_given_exp,candidate_given_exp,Penality Given,Global Similarity Given,Final Similarity Given,desired_then_exp,candidate_then_exp,Penality Then,...,intersection_then_vars,given_plus,given_less,then_plus,then_less,#given_plus,#given_less,#then_plus,#then_less,tipo_de_erro
0,Desired_1,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,height >= 30,height >= 30 AND source_distance == 0,0.090909,...,height,,"satellite_count, wind_speed",,source_distance,0,2,0,1,alfa
1,Desired_2,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,source_distance == 0,height >= 30 AND source_distance == 0,0.090909,...,source_distance,,"satellite_count, wind_speed",,height,0,2,0,1,alfa
2,Desired_3,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,a > 10,height >= 30 AND source_distance == 0,1.000000,...,,,"satellite_count, wind_speed",a,"source_distance, height",0,2,1,2,alfa+beta
3,Desired_4,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,b > 10,height >= 30 AND source_distance == 0,1.000000,...,,,"satellite_count, wind_speed",b,"source_distance, height",0,2,1,2,alfa+beta
4,Desired_5,candidate,battery >= 10,battery >= 10 AND satellite_count >= 7 AND win...,0.166667,1.0,0.833333,height >= 30 AND source_distance == 0,height >= 30 AND source_distance == 0,0.000000,...,"source_distance, height",,"satellite_count, wind_speed",,,0,2,0,0,alfa


In [484]:
import plotly.express as px

# Criar o boxplot
fig = px.box(given_then_df, y='Scenario Similarity', 
             title='Similarity Distribution by Error Type',
             labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Type Error'})

# Exibir o gráfico
fig.show()


In [485]:
# Filtrar os valores abaixo de 0.333
filtered_data = given_then_df[given_then_df["Scenario Similarity"] < 0.333]

# Quantidade de valores abaixo de 0.333
below_threshold_count = filtered_data.shape[0]

# Total de valores no DataFrame
total_count = given_then_df.shape[0]

# Porcentagem de valores abaixo de 0.333
percentage_below_threshold = (below_threshold_count / total_count) * 100

print(f"Número de valores abaixo de 0.333: {below_threshold_count}")
print(f"Porcentagem de valores abaixo de 0.333: {percentage_below_threshold:.2f}%")


Número de valores abaixo de 0.333: 123
Porcentagem de valores abaixo de 0.333: 26.51%


In [486]:
# Filtrar os valores abaixo de 0.333
filtered_data = given_then_df[given_then_df["Scenario Similarity"] >= 0.666]

# Quantidade de valores abaixo de 0.333
below_threshold_count = filtered_data.shape[0]

# Total de valores no DataFrame
total_count = given_then_df.shape[0]

# Porcentagem de valores abaixo de 0.333
percentage_below_threshold = (below_threshold_count / total_count) * 100

print(f"Número de valores acima de 0.666: {below_threshold_count}")
print(f"Porcentagem de valores acima de 0.666: {percentage_below_threshold:.2f}%")


Número de valores acima de 0.666: 270
Porcentagem de valores acima de 0.666: 58.19%


In [487]:
import plotly.express as px

# Criar o boxplot
fig = px.box(given_then_df, x='tipo_de_erro', y='Scenario Similarity', 
             title='Similarity Distribution by Error Type',
             labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Type Error'})

# Exibir o gráfico
fig.show()


In [488]:
import plotly.express as px

given_then_df['tipo_de_erro'] = given_then_df['tipo_de_erro'].replace({
    'alfa': 'Missing',
    'beta': 'Excessive',
    'alfa+beta': 'Combined'
})

# Criar o boxplot com ajustes estilizados
fig = px.box(
    given_then_df, 
    x='tipo_de_erro', 
    y='Scenario Similarity',
    labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Error Type'},
    color_discrete_sequence=["#1f77b4"]  # Cor uniforme e discreta
)

# Ajustar título e layout
fig.update_layout(
    title=None,  # Remover título acima do gráfico
    xaxis_title="Error Type",  # Legenda do eixo X
    yaxis_title="Scenario Similarity",  # Legenda do eixo Y
    font=dict(size=12),  # Ajustar tamanho da fonte
    plot_bgcolor="white",  # Fundo branco para maior clareza
    xaxis=dict(
        showgrid=True, 
        gridcolor='lightgray',
        categoryorder="total descending",  # Ordenar categorias se necessário
    ),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),  # Linhas de grade discretas
    margin=dict(l=50, r=50, t=10, b=50),  # Margens ajustadas
    bargap=0.2,  # Reduzir o espaçamento horizontal entre as caixas
    width=400,  # Largura do gráfico (10 polegadas assumindo 100px por polegada)
    height=300  # Altura do gráfico (10 polegadas assumindo 100px por polegada)
)

# Exibir o gráfico
fig.show()


In [489]:
import plotly.express as px

# Criar o boxplot
fig = px.box(given_then_df, x='tipo_de_erro', y='Scenario Similarity', 
             title='Distribuição da Similaridade por Tipo de Erro',
             labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Tipo de Erro'},
             points="all")

# Mantendo a média visível e destacando o valor da mediana
fig.update_traces(marker=dict(size=4), jitter=0.3, boxpoints='all')  # Ajustes de visualização dos pontos

# Exibir o gráfico
fig.show()


In [490]:
import plotly.express as px

# Substituir os valores na coluna tipo_de_erro
given_then_df['tipo_de_erro'] = given_then_df['tipo_de_erro'].replace({
    'alfa': 'Missing',
    'beta': 'Excessive',
    'alfa+beta': 'Combined'
})

# Criar o boxplot com ajustes estilizados
fig = px.box(
    given_then_df,
    x='tipo_de_erro',
    y='Scenario Similarity',
    labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Error Type'},
    points="all",  # Mostrar todos os pontos no boxplot
    color_discrete_sequence=["#1f77b4"]  # Cor uniforme e discreta
)

# Calcular estatísticas (Q1, mediana e Q3) para cada categoria
quartiles = given_then_df.groupby('tipo_de_erro')['Scenario Similarity'].quantile([0.25, 0.5, 0.75]).unstack()

# Adicionar anotações para Q1, mediana e Q3
for error_type, stats in quartiles.iterrows():
    q1 = stats[0.25]
    median = stats[0.5]
    q3 = stats[0.75]
    
    # Adicionar anotação para Q1
    fig.add_annotation(
        x=error_type,
        y=q1,
        text=f"{q1:.2f}",
        showarrow=False,
        font=dict(size=8, color="black"),
        yshift=0,
        xshift=40  # Ajuste da posição vertical
    )
    
    # Adicionar anotação para Mediana
    fig.add_annotation(
        x=error_type,
        y=median,
        text=f"{median:.2f}",
        showarrow=False,
        font=dict(size=8, color="black"),
        yshift=0,
        xshift=40
    )
    
    # Adicionar anotação para Q3
    fig.add_annotation(
        x=error_type,
        y=q3,
        text=f"{q3:.2f}",
        showarrow=False,
        font=dict(size=8, color="black"),
        yshift=7,
        xshift=40
    )

# Ajustar layout e estilização
fig.update_layout(
    title=None,  # Remover título acima do gráfico
    xaxis_title="Error Type",  # Legenda do eixo X
    yaxis_title="Scenario Similarity",  # Legenda do eixo Y
    font=dict(size=12),  # Ajustar tamanho da fonte
    plot_bgcolor="white",  # Fundo branco para maior clareza
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        categoryorder="total descending",  # Ordenar categorias se necessário
    ),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),  # Linhas de grade discretas
    margin=dict(l=50, r=50, t=10, b=50),  # Margens ajustadas
    bargap=0.2,  # Reduzir o espaçamento horizontal entre as caixas
    width=400,  # Largura do gráfico
    height=300  # Altura do gráfico
)

# Ajustar traços para destacar os pontos e manter a mediana visível
fig.update_traces(marker=dict(size=4), jitter=0.3, boxpoints='all')  # Ajustes de visualização dos pontos

# Exibir o gráfico
fig.show()


In [491]:
import plotly.express as px

# Criar o histograma com facet_col para separar os tipos de erro
fig = px.histogram(
    given_then_df, 
    x='Scenario Similarity', 
    color='tipo_de_erro',  # Cor diferente para cada tipo de erro
    facet_col='tipo_de_erro',  # Criar uma coluna de facetas para cada tipo de erro
    nbins=100,  # Número de bins do histograma
    title='Distribuição da Similaridade por Tipo de Erro',
    labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Error Type'},
    opacity=0.75
)

# Atualizar layout
fig.update_layout(
    xaxis_title='Scenario Similarity',
    yaxis_title='Frequency',
    font=dict(size=12),
    plot_bgcolor="white",  # Fundo branco para clareza
    showlegend=False,  # Remover legenda, pois já temos as facetas
)

# Exibir o gráfico
fig.show()


In [492]:
given_then_df.columns


Index(['desired_identifier', 'cadidate_identifier', 'desired_given_exp',
       'candidate_given_exp', 'Penality Given', 'Global Similarity Given',
       'Final Similarity Given', 'desired_then_exp', 'candidate_then_exp',
       'Penality Then', 'Global Similarity Then', 'Final Similarity Then',
       'Scenario Similarity', 'desired_given_vars', 'candidate_given_vars',
       'desired_then_vars', 'candidate_then_vars', 'intersection_given_vars',
       'intersection_then_vars', 'given_plus', 'given_less', 'then_plus',
       'then_less', '#given_plus', '#given_less', '#then_plus', '#then_less',
       'tipo_de_erro'],
      dtype='object')

In [493]:
import plotly.express as px

# Criar o scatter plot
fig = px.scatter(given_then_df, x='tipo_de_erro', y='Scenario Similarity', 
                 color='tipo_de_erro', 
                 title='Distribuição da Similaridade por Tipo de Erro',
                 labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Tipo de Erro'},
                 opacity=0.6,
                     hover_data={
                     'desired_given_exp': True, 
                     'candidate_given_exp': True,
                     'Penality Given': True,
                     'Global Similarity Given': True,
                    'Final Similarity Given': True,
                     'desired_then_exp': True, 
                     'candidate_then_exp': True,
                    'Penality Then': True,
                        'Global Similarity Then': True,
                        'Final Similarity Then': True,
                        '#given_plus': True,
                          '#given_less': True,
                            '#then_plus': True, '#then_less': True
                 })  # Define uma transparência para ver sobreposição de pontos

# Exibir o gráfico
fig.show()


In [494]:
import plotly.express as px
import numpy as np

# Convertendo a coluna 'tipo_de_erro' para uma representação numérica temporária
given_then_df['tipo_de_erro_num'] = pd.factorize(given_then_df['tipo_de_erro'])[0]  # Converte para números
given_then_df['tipo_de_erro_jittered'] = given_then_df['tipo_de_erro_num'] + np.random.uniform(-0.1, 0.1, given_then_df.shape[0])

# Criando o scatter plot com jitter aplicado
fig = px.scatter(given_then_df, x='tipo_de_erro_jittered', y='Scenario Similarity', 
                 color='tipo_de_erro', 
                 title='Distribuição da Similaridade por Tipo de Erro',
                 labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro_jittered': 'Tipo de Erro'},
                 opacity=0.6,
                 hover_data={
                     'desired_given_exp': True, 
                     'candidate_given_exp': True,
                     'Penality Given': True,
                     'Global Similarity Given': True,
                     'Final Similarity Given': True,
                     'desired_then_exp': True, 
                     'candidate_then_exp': True,
                     'Penality Then': True,
                     'Global Similarity Then': True,
                     'Final Similarity Then': True,
                     '#given_plus': True,
                     '#given_less': True,
                     '#then_plus': True, 
                     '#then_less': True
                 })

# Exibir o gráfico
fig.show()


In [495]:

fig = px.violin(given_then_df, x='tipo_de_erro', y='Scenario Similarity', 
                color='tipo_de_erro', 
                title='Distribuição da Similaridade por Tipo de Erro',
                labels={'Scenario Similarity': 'Scenario Similarity', 'tipo_de_erro': 'Tipo de Erro'},
                hover_data={
                    'desired_given_exp': True, 
                    'candidate_given_exp': True,
                    'Penality Given': True,
                    'Global Similarity Given': True,
                    'Final Similarity Given': True,
                    'desired_then_exp': True, 
                    'candidate_then_exp': True,
                    'Penality Then': True,
                    'Global Similarity Then': True,
                    'Final Similarity Then': True,
                    '#given_plus': True,
                    '#given_less': True,
                    '#then_plus': True, 
                    '#then_less': True
                })

fig.show()

alfa-error: quando o derivado (desejado) é errado por falta
- a,b vs a,b,c (desejado nao tem o c)

beta-error: quando o derivado (desejado) é errado por excesso
- a,b,c vs a,b (desejado tem o c de extra)

alfa-beta-erro: quando o derivado é errado por falta e excesso
- a,b,d vs a,b,c (desejado nao tem o c e tem o d de extra)

Certamente podemos dizer sem medo que temos o threshold de 0.333, pois isso significa que ou o Given ou o Then é completamente diferente do alvo

Então temos o seguinte, dependendo do tipo de erro que pode indicar um influencia negativa simples até grave. Repare que error alfa de modelagem do desejado, nao prejudica tanto encontrar o maximo global, ja o erro beta prejudica um pouco mais, mas ainda sim pouco, mas já o erro alfa-beta pode prejudicar bastante.